In [2]:
import random 
import pandas as pd
from haversine import haversine, Unit
import numpy as np

In [3]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [4]:
from scripts.PHMM.viterbi_algorithm.states import *
from scripts.PHMM.viterbi_algorithm.viterbi import *
from scripts.PHMM.distributions.AIS_RF_distribution import *
import pandas as pd

In [7]:
begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
model = PHMM(states, end)

In [8]:
test_set = pd.read_parquet("../data/processed/PHMM_sequence_data.parquet")
test_set

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
0,0,209112000,0,2024-01-03 13:19:57,NaT,NaT,None,None,begin
1,0,209112000,0,2024-01-03 13:19:58,2024-01-03 13:19:58,NaT,"[48.51548, -126.51201]",None,AIS
2,0,209112000,0,2024-01-03 13:32:50,2024-01-03 13:32:50,NaT,"[48.51033, -126.45776]",None,AIS
3,0,209112000,0,2024-01-03 13:34:50,2024-01-03 13:34:50,NaT,"[48.50968, -126.44914]",None,AIS
4,0,209112000,0,2024-01-03 13:40:59,2024-01-03 13:40:59,2024-01-03 13:41:00,"[48.50753, -126.42323]","[48.499443942360536, -126.4146031322306]",M
...,...,...,...,...,...,...,...,...,...
471750,710,677059800,0,2024-01-08 08:30:39,2024-01-08 08:30:39,NaT,"[19.00953, -68.1877]",None,AIS
471751,710,677059800,0,2024-01-08 08:47:23,2024-01-08 08:47:23,NaT,"[19.02076, -68.22079]",None,AIS
471752,710,677059800,0,2024-01-08 08:50:39,2024-01-08 08:50:39,NaT,"[19.02278, -68.22751]",None,AIS
471753,710,677059800,0,2024-01-08 09:41:00,NaT,2024-01-08 09:41:00,None,"[19.024754315587664, -68.23688644056789]",RF


In [9]:
sequence = test_set[test_set['TrackID'] == 5]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence = [(time, coord) for time, coord in zip(sequence["RF_Timestamp"], sequence["RF"]) 
               if pd.notna(time) and coord is not None][1:2]

In [10]:
RF_sequence

[(Timestamp('2024-01-03 09:41:00'), array([ 36.91790966, -75.90767514]))]

In [11]:
path, probability, log_probs = model.viterbi(AIS_sequence, RF_sequence)

[RESET] gap_counter = 0
0
0
2
4
4
4
6
8
10
10
10
12
14
16
16
16
18
20
22
22
22
24
26
28
28
28
30
32
34
34
34
36
38
40
40
40
42
44
46
46
46
48
50
52
52
52
54
56
58
58
58
60
62
64
64
64
66
68
70
70
70
72
74
76
76
76
78
80
82
82
82
84
86
88
88
88
90
92
94
94
94
96
98
100
100
100
102
104
106
106
106
108
110
112
112
112
114
116
118
118
118
120
122
124
124
124
126
128
130
130
130
132
134
136
136
136
138
140
142
142
142
144
146
148
148
148
150
152
154
154
154
156
158
160
160
160
162
164
166
166
166
168
170
172
172
172
174
176
178
178
178
180
182
184
184
184
186
188
190
190
190
192
194
196
196
196
198
200
202
202
202
204
206
208
208
208
210
212
214
214
214
216
218
220
220
220
222
224
226
226
226
228
230
232
232
232
234
236
238
238
238
240
242
244
244
244
246
248
250
250
250
252
254
256
256
256
258
260
262
262
262
264
266
268
268
268
270
272
274
274
274
276
278
280
280
280
282
284
286
286
286
288
290
292
292
292
294
296
298
298
298
300
302
304
304
304
306
308
310
310
310
312
314
316
316
316
318

In [12]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(219, 'RF')],       
The indexes of RF and/or M of the real path are: [(34, 'RF'), (220, 'M'), (350, 'M'), (544, 'RF')]


In [13]:
print(probability)
print(probability/len(path))

-27.615797928507973
-0.05067118885964766


In [17]:
df = test_set[test_set["TrackID"] == 5].copy()
match_rows = df[df["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

rf_from_match = match_rows.copy()
rf_from_match["AIS"] = None
rf_from_match["AIS_Timestamp"] = pd.NaT
rf_from_match["Timestamp"] = rf_from_match["RF_Timestamp"]
rf_from_match["State"] = "RF"

rf_from_original = df[df["State"] == "RF"]
all_rf_candidates = pd.concat([rf_from_original, rf_from_match], ignore_index=True)
chosen_rf = all_rf_candidates.sort_values(by="RF_Timestamp")[1:2]

df = df[(df["State"] != "M") & (df["State"] != "RF")]
df = pd.concat([df, ais_rows, chosen_rf], ignore_index=True)
df = df.sort_values(by="Timestamp").reset_index(drop=True)

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged = df.copy().reset_index(drop=True)
df_merged["BestPathState"] = df_path["State"]
df_merged["LogProb"] = df_path["LogProb"]
df_merged["Probability"] = np.exp(df_path["LogProb"])

df_merged["prev_Timestamp"] = df_merged["Timestamp"].shift(1)
df_merged["delta_time_sec"] = (df_merged["Timestamp"] - df_merged["prev_Timestamp"]).dt.total_seconds()

df_merged["coord"] = df_merged["AIS"].combine_first(df_merged["RF"])
df_merged["prev_coord"] = df_merged["coord"].shift(1)
df_merged["distance_km"] = None  

for idx, row in df_merged.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged.at[idx, "distance_km"] = dist
        
df_merged["prev_state"] = df_merged["State"].shift(1)
df_merged["transition"] = df_merged["prev_state"] + "→" + df_merged["State"]
df_merged["trans_prob"] = np.nan

for idx, row in df_merged.iterrows():
    trans = row["transition"]

    if row["prev_state"] == "begin":
        trans_prob = 0.3

    elif trans in ["AIS→AIS", "AIS→RF", "RF→RF", "RF→AIS"]:
        prev = df_merged.loc[idx - 1]

        if trans == "AIS→AIS":
            time = (row["AIS_Timestamp"] - prev["AIS_Timestamp"]).total_seconds()
            dist = haversine(row["AIS"], prev["AIS"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "AIS→RF":
            time = (row["RF_Timestamp"] - prev["AIS_Timestamp"]).total_seconds()
            dist = haversine(row["RF"], prev["AIS"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "RF→RF":
            time = (row["RF_Timestamp"] - prev["RF_Timestamp"]).total_seconds()
            dist = haversine(row["RF"], prev["RF"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "RF→AIS":
            time = (row["AIS_Timestamp"] - prev["RF_Timestamp"]).total_seconds()
            dist = haversine(row["AIS"], prev["RF"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)

    elif row["State"] == "end":
        trans_prob = 0.05
    
    else:
        continue

    df_merged.at[idx, "trans_prob"] = trans_prob

df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
0,5,210614000,2,2024-01-03 00:03:02,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None,None,NaN,NaN
1,5,210614000,2,2024-01-03 00:03:03,2024-01-03 00:03:03,NaT,"[36.96283, -76.42414]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:03:02,1.0,"[36.96283, -76.42414]",None,None,begin,begin→AIS,0.300000
2,5,210614000,2,2024-01-03 00:06:03,2024-01-03 00:06:03,NaT,"[36.96283, -76.42412]",None,AIS,AIS,-1.127221,3.239323e-01,2024-01-03 00:03:03,180.0,"[36.96283, -76.42412]","[36.96283, -76.42414]",0.001777,AIS,AIS→AIS,0.981828
3,5,210614000,2,2024-01-03 00:09:03,2024-01-03 00:09:03,NaT,"[36.96282, -76.42411]",None,AIS,AIS,-1.153941,3.153913e-01,2024-01-03 00:06:03,180.0,"[36.96282, -76.42411]","[36.96283, -76.42412]",0.001423,AIS,AIS→AIS,0.981880
4,5,210614000,2,2024-01-03 00:12:03,2024-01-03 00:12:03,NaT,"[36.96282, -76.42409]",None,AIS,AIS,-1.180245,3.072034e-01,2024-01-03 00:09:03,180.0,"[36.96282, -76.42409]","[36.96282, -76.42411]",0.001777,AIS,AIS→AIS,0.981828
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,5,210614000,2,2024-01-03 20:35:06,2024-01-03 20:35:06,NaT,"[36.04509, -73.8082]",None,AIS,AIS,-24.543126,2.193087e-11,2024-01-03 20:12:17,1369.0,"[36.04509, -73.8082]","[36.0808, -73.89056]",8.400754,AIS,AIS→AIS,0.247199
541,5,210614000,2,2024-01-03 20:39:06,2024-01-03 20:39:06,NaT,"[36.03897, -73.79441]",None,AIS,AIS,-24.568772,2.137557e-11,2024-01-03 20:35:06,240.0,"[36.03897, -73.79441]","[36.04509, -73.8082]",1.414346,AIS,AIS→AIS,0.789585
542,5,210614000,2,2024-01-03 20:44:27,2024-01-03 20:44:27,NaT,"[36.03039, -73.77547]",None,AIS,AIS,-24.594419,2.083433e-11,2024-01-03 20:39:06,321.0,"[36.03039, -73.77547]","[36.03897, -73.79441]",1.952091,AIS,AIS→AIS,0.722497
543,5,210614000,2,2024-01-03 21:07:57,2024-01-03 21:07:57,NaT,"[35.99436, -73.69102]",None,AIS,AIS,-24.620066,2.030679e-11,2024-01-03 20:44:27,1410.0,"[35.99436, -73.69102]","[36.03039, -73.77547]",8.58763,AIS,AIS→AIS,0.239378


In [19]:
df_merged.sort_values(by=["delta_time_sec"], ascending=False)

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
107,5,210614000,2,2024-01-03 07:32:37,2024-01-03 07:32:37,NaT,"[36.98653, -76.31608]",None,AIS,AIS,-4.376353,1.257112e-02,2024-01-03 05:18:03,8074.0,"[36.98653, -76.31608]","[36.96283, -76.42408]",9.949419,AIS,AIS→AIS,0.010000
445,5,210614000,2,2024-01-03 17:02:27,2024-01-03 17:02:27,NaT,"[36.34553, -74.54543]",None,AIS,AIS,-22.106694,2.507175e-10,2024-01-03 15:30:57,5490.0,"[36.34553, -74.54543]","[36.47445, -74.84854]",30.679964,AIS,AIS→AIS,0.010000
298,5,210614000,2,2024-01-03 12:40:47,2024-01-03 12:40:47,NaT,"[36.69973, -75.38304]",None,AIS,AIS,-18.336637,1.087676e-08,2024-01-03 11:10:36,5411.0,"[36.69973, -75.38304]","[36.77195, -75.6805]",27.69696,AIS,AIS→AIS,0.010000
544,5,210614000,2,2024-01-03 21:41:01,NaT,NaT,None,None,end,end,-27.615798,1.015340e-12,2024-01-03 21:07:57,1984.0,None,"[35.99436, -73.69102]",None,AIS,AIS→end,0.050000
543,5,210614000,2,2024-01-03 21:07:57,2024-01-03 21:07:57,NaT,"[35.99436, -73.69102]",None,AIS,AIS,-24.620066,2.030679e-11,2024-01-03 20:44:27,1410.0,"[35.99436, -73.69102]","[36.03039, -73.77547]",8.58763,AIS,AIS→AIS,0.239378
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,5,210614000,2,2024-01-03 09:41:00,NaT,2024-01-03 09:41:00,None,"[36.917909658337884, -75.90767513607126]",RF,RF,-16.310552,8.249304e-08,2024-01-03 09:40:27,33.0,"[36.917909658337884, -75.90767513607126]","[36.91522, -75.91979]",1.117786,AIS,AIS→RF,0.842838
183,5,210614000,2,2024-01-03 09:00:07,2024-01-03 09:00:07,NaT,"[36.95467, -76.0385]",None,AIS,AIS,-7.301911,6.742494e-04,2024-01-03 08:59:36,31.0,"[36.95467, -76.0385]","[36.95503, -76.04017]",0.153696,AIS,AIS→AIS,0.974173
237,5,210614000,2,2024-01-03 10:00:07,2024-01-03 10:00:07,NaT,"[36.87622, -75.87551]",None,AIS,AIS,-16.772192,5.199119e-08,2024-01-03 09:59:57,10.0,"[36.87622, -75.87551]","[36.87652, -75.87591]",0.048772,AIS,AIS→AIS,0.991715
1,5,210614000,2,2024-01-03 00:03:03,2024-01-03 00:03:03,NaT,"[36.96283, -76.42414]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:03:02,1.0,"[36.96283, -76.42414]",None,None,begin,begin→AIS,0.300000


In [15]:
df_merged[df_merged["transition"] == "AIS→RF"]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
219,5,210614000,2,2024-01-03 09:41:00,NaT,2024-01-03 09:41:00,None,"[36.917909658337884, -75.90767513607126]",RF,RF,-16.310552,8.249304e-08,2024-01-03 09:40:27,33.0,"[36.917909658337884, -75.90767513607126]","[36.91522, -75.91979]",1.117786,AIS,AIS→RF,0.842838


In [16]:
df_merged.sort_values(by=["trans_prob"], inplace=True)
df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
445,5,210614000,2,2024-01-03 17:02:27,2024-01-03 17:02:27,NaT,"[36.34553, -74.54543]",None,AIS,AIS,-22.106694,2.507175e-10,2024-01-03 15:30:57,5490.0,"[36.34553, -74.54543]","[36.47445, -74.84854]",30.679964,AIS,AIS→AIS,0.010000
298,5,210614000,2,2024-01-03 12:40:47,2024-01-03 12:40:47,NaT,"[36.69973, -75.38304]",None,AIS,AIS,-18.336637,1.087676e-08,2024-01-03 11:10:36,5411.0,"[36.69973, -75.38304]","[36.77195, -75.6805]",27.69696,AIS,AIS→AIS,0.010000
107,5,210614000,2,2024-01-03 07:32:37,2024-01-03 07:32:37,NaT,"[36.98653, -76.31608]",None,AIS,AIS,-4.376353,1.257112e-02,2024-01-03 05:18:03,8074.0,"[36.98653, -76.31608]","[36.96283, -76.42408]",9.949419,AIS,AIS→AIS,0.010000
544,5,210614000,2,2024-01-03 21:41:01,NaT,NaT,None,None,end,end,-27.615798,1.015340e-12,2024-01-03 21:07:57,1984.0,None,"[35.99436, -73.69102]",None,AIS,AIS→end,0.050000
543,5,210614000,2,2024-01-03 21:07:57,2024-01-03 21:07:57,NaT,"[35.99436, -73.69102]",None,AIS,AIS,-24.620066,2.030679e-11,2024-01-03 20:44:27,1410.0,"[35.99436, -73.69102]","[36.03039, -73.77547]",8.58763,AIS,AIS→AIS,0.239378
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,5,210614000,2,2024-01-03 04:33:03,2024-01-03 04:33:03,NaT,"[36.96278, -76.42407]",None,AIS,AIS,-3.417226,3.280330e-02,2024-01-03 04:30:03,180.0,"[36.96278, -76.42407]","[36.96278, -76.42407]",0.0,AIS,AIS→AIS,0.982090
72,5,210614000,2,2024-01-03 03:36:03,2024-01-03 03:36:03,NaT,"[36.96278, -76.42409]",None,AIS,AIS,-2.929538,5.342173e-02,2024-01-03 03:33:03,180.0,"[36.96278, -76.42409]","[36.96278, -76.42409]",0.0,AIS,AIS→AIS,0.982090
78,5,210614000,2,2024-01-03 03:54:03,2024-01-03 03:54:03,NaT,"[36.96278, -76.42409]",None,AIS,AIS,-3.083555,4.579615e-02,2024-01-03 03:51:03,180.0,"[36.96278, -76.42409]","[36.96278, -76.42409]",0.0,AIS,AIS→AIS,0.982090
237,5,210614000,2,2024-01-03 10:00:07,2024-01-03 10:00:07,NaT,"[36.87622, -75.87551]",None,AIS,AIS,-16.772192,5.199119e-08,2024-01-03 09:59:57,10.0,"[36.87622, -75.87551]","[36.87652, -75.87591]",0.048772,AIS,AIS→AIS,0.991715


In [17]:
sequence = test_set[test_set['TrackID'] == 79]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

In [18]:
path, probability, log_probs = model.viterbi(AIS_sequence, RF_sequence)

In [19]:
print(probability)
print(probability/len(path))

-34.59862879259538
-0.052263789716911445


In [18]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(378, 'RF')],       
The indexes of RF and/or M of the real path are: [(1, 'RF'), (505, 'M')]


In [19]:
df_rf = test_set[test_set["TrackID"] == 5].copy()
match_rf_rows = df_rf[df_rf["State"] == "M"]

rf_from_match = match_rf_rows.copy()
rf_from_match["AIS"] = None
rf_from_match["AIS_Timestamp"] = pd.NaT
rf_from_match["Timestamp"] = rf_from_match["RF_Timestamp"]
rf_from_match["State"] = "RF"

rf_from_original = df_rf[df_rf["State"] == "RF"]
all_rf_candidates = pd.concat([rf_from_original, rf_from_match], ignore_index=True)
chosen_rf = all_rf_candidates.sort_values(by="RF_Timestamp").iloc[1:2]

df_ais = test_set[test_set["TrackID"] == 79].copy()
match_ais_rows = df_ais[df_ais["State"] == "M"]

ais_rows = match_ais_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

df = df_ais[(df_ais["State"] != "M") & (df_ais["State"] != "RF")]
combined_df = pd.concat([df, ais_rows, chosen_rf], ignore_index=True)
combined_df = combined_df.sort_values(by="Timestamp").reset_index(drop=True)

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged2 = combined_df.copy().reset_index(drop=True)
df_merged2["BestPathState"] = df_path["State"]
df_merged2["LogProb"] = df_path["LogProb"]
df_merged2["Probability"] = np.exp(df_path["LogProb"])

df_merged2["prev_Timestamp"] = df_merged2["Timestamp"].shift(1)
df_merged2["delta_time_sec"] = (df_merged2["Timestamp"] - df_merged2["prev_Timestamp"]).dt.total_seconds()

df_merged2["coord"] = df_merged2["AIS"].combine_first(df_merged2["RF"])
df_merged2["prev_coord"] = df_merged2["coord"].shift(1)
df_merged2["distance_km"] = None  

for idx, row in df_merged2.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged2.at[idx, "distance_km"] = dist

df_merged2["prev_state"] = df_merged2["State"].shift(1)
df_merged2["transition"] = df_merged2["prev_state"] + "→" + df_merged2["State"]
df_merged2["trans_prob"] = np.nan

for idx, row in df_merged2.iterrows():
    trans = row["transition"]

    if row["prev_state"] == "begin":
        trans_prob = 0.3

    elif trans in ["AIS→AIS", "AIS→RF", "RF→RF", "RF→AIS"]:
        prev = df_merged2.loc[idx - 1]

        if trans == "AIS→AIS":
            time = (row["AIS_Timestamp"] - prev["AIS_Timestamp"]).total_seconds()
            dist = haversine(row["AIS"], prev["AIS"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "AIS→RF":
            time = (row["RF_Timestamp"] - prev["AIS_Timestamp"]).total_seconds()
            dist = haversine(row["RF"], prev["AIS"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "RF→RF":
            time = (row["RF_Timestamp"] - prev["RF_Timestamp"]).total_seconds()
            dist = haversine(row["RF"], prev["RF"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "RF→AIS":
            time = (row["AIS_Timestamp"] - prev["RF_Timestamp"]).total_seconds()
            dist = haversine(row["AIS"], prev["RF"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
                
    elif row["State"] == "end":
        trans_prob = 0.05
    
    else:
        continue

    df_merged2.at[idx, "trans_prob"] = trans_prob

df_merged2

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
0,79,247310400,2,2024-01-02 21:40:59,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None,None,NaN,NaN
1,79,247310400,2,2024-01-02 23:59:33,2024-01-02 23:59:33,NaT,"[39.1948, -76.48997]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-02 21:40:59,8314.0,"[39.1948, -76.48997]",None,None,begin,begin→AIS,0.300000
2,79,247310400,2,2024-01-03 00:00:03,2024-01-03 00:00:03,NaT,"[39.19433, -76.48838]",None,AIS,AIS,-1.134417,3.216096e-01,2024-01-02 23:59:33,30.0,"[39.19433, -76.48838]","[39.1948, -76.48997]",0.146649,AIS,AIS→AIS,0.974470
3,79,247310400,2,2024-01-03 00:02:53,2024-01-03 00:02:53,NaT,"[39.19167, -76.47924]",None,AIS,AIS,-1.171525,3.098941e-01,2024-01-03 00:00:03,170.0,"[39.19167, -76.47924]","[39.19433, -76.48838]",0.841376,AIS,AIS→AIS,0.862567
4,79,247310400,2,2024-01-03 00:04:04,2024-01-03 00:04:04,NaT,"[39.19053, -76.47539]",None,AIS,AIS,-1.207702,2.988835e-01,2024-01-03 00:02:53,71.0,"[39.19053, -76.47539]","[39.19167, -76.47924]",0.355187,AIS,AIS→AIS,0.939676
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
657,79,247310400,2,2024-01-03 17:02:27,2024-01-03 17:02:27,NaT,"[35.57177, -74.40962]",None,AIS,AIS,-31.831891,1.498254e-14,2024-01-03 17:00:27,120.0,"[35.57177, -74.40962]","[35.57831, -74.41664]",0.965368,AIS,AIS→AIS,0.860027
658,79,247310400,2,2024-01-03 17:03:45,2024-01-03 17:03:45,NaT,"[35.56753, -74.40505]",None,AIS,AIS,-31.857537,1.460318e-14,2024-01-03 17:02:27,78.0,"[35.56753, -74.40505]","[35.57177, -74.40962]",0.627004,AIS,AIS→AIS,0.906691
659,79,247310400,2,2024-01-03 17:05:09,2024-01-03 17:05:09,NaT,"[35.56301, -74.40028]",None,AIS,AIS,-31.883184,1.423342e-14,2024-01-03 17:03:45,84.0,"[35.56301, -74.40028]","[35.56753, -74.40505]",0.662392,AIS,AIS→AIS,0.901330
660,79,247310400,2,2024-01-03 17:11:21,2024-01-03 17:11:21,NaT,"[35.54379, -74.37878]",None,AIS,AIS,-31.908830,1.387302e-14,2024-01-03 17:05:09,372.0,"[35.54379, -74.37878]","[35.56301, -74.40028]",2.889731,AIS,AIS→AIS,0.634708


In [20]:
test_set[test_set["TrackID"] == 79]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
48252,79,247310400,2,2024-01-02 21:40:59,NaT,NaT,None,None,begin
48253,79,247310400,2,2024-01-02 21:41:00,NaT,2024-01-02 21:41:00,None,"[39.192009451935135, -76.4795149778899]",RF
48254,79,247310400,2,2024-01-02 23:59:33,2024-01-02 23:59:33,NaT,"[39.1948, -76.48997]",None,AIS
48255,79,247310400,2,2024-01-03 00:00:03,2024-01-03 00:00:03,NaT,"[39.19433, -76.48838]",None,AIS
48256,79,247310400,2,2024-01-03 00:02:53,2024-01-03 00:02:53,NaT,"[39.19167, -76.47924]",None,AIS
...,...,...,...,...,...,...,...,...,...
48909,79,247310400,2,2024-01-03 17:02:27,2024-01-03 17:02:27,NaT,"[35.57177, -74.40962]",None,AIS
48910,79,247310400,2,2024-01-03 17:03:45,2024-01-03 17:03:45,NaT,"[35.56753, -74.40505]",None,AIS
48911,79,247310400,2,2024-01-03 17:05:09,2024-01-03 17:05:09,NaT,"[35.56301, -74.40028]",None,AIS
48912,79,247310400,2,2024-01-03 17:11:21,2024-01-03 17:11:21,NaT,"[35.54379, -74.37878]",None,AIS


In [22]:
df_merged2[df_merged2["transition"] == "AIS→RF"]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
378,5,210614000,2,2024-01-03 09:41:00,NaT,2024-01-03 09:41:00,None,"[36.917909658337884, -75.90767513607126]",RF,RF,-24.676476,1.919299e-11,2024-01-03 09:40:22,38.0,"[36.917909658337884, -75.90767513607126]","[36.88085, -75.89407]",4.294766,AIS,AIS→RF,0.57904


In [21]:
df_merged2.sort_values(by=["trans_prob"], inplace=True)
df_merged2

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
436,79,247310400,2,2024-01-03 12:25:45,2024-01-03 12:25:45,NaT,"[36.44869, -75.37019]",None,AIS,AIS,-26.163982,4.336379e-12,2024-01-03 10:46:33,5952.0,"[36.44869, -75.37019]","[36.77036, -75.71237]",47.033926,AIS,AIS→AIS,0.010000
158,79,247310400,2,2024-01-03 05:36:10,2024-01-03 05:36:10,NaT,"[37.76549, -76.18694]",None,AIS,AIS,-7.351033,6.419287e-04,2024-01-03 02:56:45,9565.0,"[37.76549, -76.18694]","[38.50747, -76.40766]",84.732694,AIS,AIS→AIS,0.010000
661,79,247310400,2,2024-01-03 17:11:22,NaT,NaT,None,None,end,end,-34.904563,6.936510e-16,2024-01-03 17:11:21,1.0,None,"[35.54379, -74.37878]",None,AIS,AIS→end,0.050000
1,79,247310400,2,2024-01-02 23:59:33,2024-01-02 23:59:33,NaT,"[39.1948, -76.48997]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-02 21:40:59,8314.0,"[39.1948, -76.48997]",None,None,begin,begin→AIS,0.300000
637,79,247310400,2,2024-01-03 16:29:46,2024-01-03 16:29:46,NaT,"[35.68175, -74.51933]",None,AIS,AIS,-31.318958,2.502359e-14,2024-01-03 16:20:45,541.0,"[35.68175, -74.51933]","[35.71183, -74.54945]",4.311078,AIS,AIS→AIS,0.509320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
468,79,247310400,2,2024-01-03 13:00:04,2024-01-03 13:00:04,NaT,"[36.34853, -75.24576]",None,AIS,AIS,-26.984674,1.908556e-12,2024-01-03 12:59:39,25.0,"[36.34853, -75.24576]","[36.34978, -75.24725]",0.192682,AIS,AIS→AIS,0.970096
2,79,247310400,2,2024-01-03 00:00:03,2024-01-03 00:00:03,NaT,"[39.19433, -76.48838]",None,AIS,AIS,-1.134417,3.216096e-01,2024-01-02 23:59:33,30.0,"[39.19433, -76.48838]","[39.1948, -76.48997]",0.146649,AIS,AIS→AIS,0.974470
523,79,247310400,2,2024-01-03 14:00:04,2024-01-03 14:00:04,NaT,"[36.16839, -75.02775]",None,AIS,AIS,-28.395240,4.656976e-13,2024-01-03 13:59:45,19.0,"[36.16839, -75.02775]","[36.16933, -75.02895]",0.150095,AIS,AIS→AIS,0.976744
396,79,247310400,2,2024-01-03 10:00:04,2024-01-03 10:00:04,NaT,"[36.84757, -75.84144]",None,AIS,AIS,-25.138116,1.209637e-11,2024-01-03 09:59:53,11.0,"[36.84757, -75.84144]","[36.84784, -75.84198]",0.056658,AIS,AIS→AIS,0.990205


In [22]:
sequence = test_set[test_set['TrackID'] == 197]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

In [23]:
path, probability, log_probs = model.viterbi(AIS_sequence, RF_sequence)

In [24]:
print(probability)
print(probability/len(path))

-17.69218727503968
-0.05026189566772636


In [25]:
RF_M_indexes_pred_path = [(i, state) for i, state in enumerate(path) if state in {"RF", "M"}]
RF_M_indexes_real_path = [(i, state) for i, state in enumerate(sequence["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path}")

The indexes of RF and/or M of predicted path are: [(73, 'RF')],       
The indexes of RF and/or M of the real path are: [(73, 'M'), (322, 'RF')]


In [ ]:
df_rf = test_set[test_set["TrackID"] == 46].copy()
match_rf_rows = df_rf[df_rf["State"] == "M"]

rf_from_match = match_rf_rows.copy()
rf_from_match["AIS"] = None
rf_from_match["AIS_Timestamp"] = pd.NaT
rf_from_match["Timestamp"] = rf_from_match["RF_Timestamp"]
rf_from_match["State"] = "RF"

rf_from_original = df_rf[df_rf["State"] == "RF"]
all_rf_candidates = pd.concat([rf_from_original, rf_from_match], ignore_index=True)
chosen_rf = all_rf_candidates.sort_values(by="RF_Timestamp").iloc[2:3]

df_ais = test_set[test_set["TrackID"] == 197].copy()
match_ais_rows = df_ais[df_ais["State"] == "M"]

ais_rows = match_ais_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

df = df_ais[(df_ais["State"] != "M") & (df_ais["State"] != "RF")]
combined_df = pd.concat([df, ais_rows, chosen_rf], ignore_index=True)
combined_df = combined_df.sort_values(by="Timestamp").reset_index(drop=True)

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged2 = combined_df.copy().reset_index(drop=True)
df_merged2["BestPathState"] = df_path["State"]
df_merged2["LogProb"] = df_path["LogProb"]
df_merged2["Probability"] = np.exp(df_path["LogProb"])

df_merged2["prev_Timestamp"] = df_merged2["Timestamp"].shift(1)
df_merged2["delta_time_sec"] = (df_merged2["Timestamp"] - df_merged2["prev_Timestamp"]).dt.total_seconds()

df_merged2["coord"] = df_merged2["AIS"].combine_first(df_merged2["RF"])
df_merged2["prev_coord"] = df_merged2["coord"].shift(1)
df_merged2["distance_km"] = None  

for idx, row in df_merged2.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged2.at[idx, "distance_km"] = dist

df_merged2["prev_state"] = df_merged2["State"].shift(1)
df_merged2["transition"] = df_merged2["prev_state"] + "→" + df_merged2["State"]
df_merged2["trans_prob"] = np.nan

gap_counter = 0

for idx, row in df_merged2.iterrows():
    trans = row["transition"]

    if row["prev_state"] == "begin":
        trans_prob = 0.3

    elif trans in ["AIS→AIS", "AIS→RF", "RF→RF", "RF→AIS"]:
        prev = df_merged2.loc[idx - 1]

        if trans == "AIS→AIS":
            time = (row["AIS_Timestamp"] - prev["AIS_Timestamp"]).total_seconds()
            dist = haversine(row["AIS"], prev["AIS"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "AIS→RF":
            time = (row["RF_Timestamp"] - prev["AIS_Timestamp"]).total_seconds()
            dist = haversine(row["RF"], prev["AIS"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
                gap_counter += 1
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "RF→RF":
            time = (row["RF_Timestamp"] - prev["RF_Timestamp"]).total_seconds()
            dist = haversine(row["RF"], prev["RF"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "RF→AIS":
            time = (row["AIS_Timestamp"] - prev["RF_Timestamp"]).total_seconds()
            dist = haversine(row["AIS"], prev["RF"], unit=Unit.KILOMETERS)
            if time >= 5400:
                trans_prob = 0.01
            else:
                trans_prob = AIS_RF_probability(time, dist)
                
    elif row["State"] == "end":
        trans_prob = 0.05
    
    else:
        continue

    df_merged2.at[idx, "trans_prob"] = trans_prob

df_merged2

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
0,197,352002625,3,2024-01-25 03:23:05,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None,None,NaN,NaN
1,197,352002625,3,2024-01-25 03:23:06,2024-01-25 03:23:06,NaT,"[33.71173, -118.20001]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-25 03:23:05,1.0,"[33.71173, -118.20001]",None,None,begin,begin→AIS,0.300000
2,197,352002625,3,2024-01-25 03:26:06,2024-01-25 03:26:06,NaT,"[33.71167, -118.20008]",None,AIS,AIS,-1.135688,3.212010e-01,2024-01-25 03:23:06,180.0,"[33.71167, -118.20008]","[33.71173, -118.20001]",0.009297,AIS,AIS→AIS,0.864980
3,197,352002625,3,2024-01-25 03:29:06,2024-01-25 03:29:06,NaT,"[33.71158, -118.20017]",None,AIS,AIS,-1.172767,3.095094e-01,2024-01-25 03:26:06,180.0,"[33.71158, -118.20017]","[33.71167, -118.20008]",0.013017,AIS,AIS→AIS,0.864771
4,197,352002625,3,2024-01-25 03:32:06,2024-01-25 03:32:06,NaT,"[33.7115, -118.20021]",None,AIS,AIS,-1.209843,2.982440e-01,2024-01-25 03:29:06,180.0,"[33.7115, -118.20021]","[33.71158, -118.20017]",0.009634,AIS,AIS→AIS,0.864961
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,197,352002625,3,2024-01-26 03:09:27,2024-01-26 03:09:27,NaT,"[33.72958, -118.25682]",None,AIS,AIS,-14.619515,4.475332e-07,2024-01-26 03:06:27,180.0,"[33.72958, -118.25682]","[33.72961, -118.25675]",0.007282,AIS,AIS→AIS,0.865093
348,197,352002625,3,2024-01-26 03:12:27,2024-01-26 03:12:27,NaT,"[33.72962, -118.25679]",None,AIS,AIS,-14.645162,4.362014e-07,2024-01-26 03:09:27,180.0,"[33.72962, -118.25679]","[33.72958, -118.25682]",0.005242,AIS,AIS→AIS,0.865208
349,197,352002625,3,2024-01-26 03:15:27,2024-01-26 03:15:27,NaT,"[33.72958, -118.25677]",None,AIS,AIS,-14.670808,4.251566e-07,2024-01-26 03:12:27,180.0,"[33.72958, -118.25677]","[33.72962, -118.25679]",0.004817,AIS,AIS→AIS,0.865232
350,197,352002625,3,2024-01-26 03:18:27,2024-01-26 03:18:27,NaT,"[33.7296, -118.2568]",None,AIS,AIS,-14.696455,4.143914e-07,2024-01-26 03:15:27,180.0,"[33.7296, -118.2568]","[33.72958, -118.25677]",0.003556,AIS,AIS→AIS,0.865303


In [16]:
df = test_set[test_set["TrackID"] == 164]
match_rows = df[df["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

rf_rows = match_rows.copy()
rf_rows["AIS"] = None
rf_rows["AIS_Timestamp"] = pd.NaT
rf_rows["Timestamp"] = rf_rows["RF_Timestamp"]
rf_rows["State"] = "RF"

df = df[(df["State"] != "M") & (df["State"] != "RF")]
df = pd.concat([df, ais_rows, rf_rows.iloc[0:1]], ignore_index=True)
df = df.sort_values(by=["Timestamp"]).reset_index(drop=True) 

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged = df.copy().reset_index(drop=True)
df_merged["BestPathState"] = df_path["State"]
df_merged["LogProb"] = df_path["LogProb"]
df_merged["Probability"] = np.exp(df_path["LogProb"])

df_merged["prev_Timestamp"] = df_merged["Timestamp"].shift(1)
df_merged["delta_time_sec"] = (df_merged["Timestamp"] - df_merged["prev_Timestamp"]).dt.total_seconds()

df_merged["coord"] = df_merged["AIS"].combine_first(df_merged["RF"])
df_merged["prev_coord"] = df_merged["coord"].shift(1)
df_merged["distance_km"] = None  

for idx, row in df_merged.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged.at[idx, "distance_km"] = dist
df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km
0,164,316009457,2,2024-01-03 00:01:08,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None
1,164,316009457,2,2024-01-03 00:01:09,2024-01-03 00:01:09,NaT,"[43.26082, -82.44656]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:01:08,1.0,"[43.26082, -82.44656]",None,None
2,164,316009457,2,2024-01-03 00:02:59,2024-01-03 00:02:59,NaT,"[43.25506, -82.44536]",None,AIS,AIS,-1.124278,3.248868e-01,2024-01-03 00:01:09,110.0,"[43.25506, -82.44536]","[43.26082, -82.44656]",0.647814
3,164,316009457,2,2024-01-03 00:04:09,2024-01-03 00:04:09,NaT,"[43.2514, -82.44465]",None,AIS,AIS,-1.149946,3.166538e-01,2024-01-03 00:02:59,70.0,"[43.2514, -82.44465]","[43.25506, -82.44536]",0.411016
4,164,316009457,2,2024-01-03 00:05:20,2024-01-03 00:05:20,NaT,"[43.24775, -82.4439]",None,AIS,AIS,-1.175615,3.086291e-01,2024-01-03 00:04:09,71.0,"[43.24775, -82.4439]","[43.2514, -82.44465]",0.410383
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
917,164,316009457,2,2024-01-03 23:56:49,2024-01-03 23:56:49,NaT,"[42.26315, -83.10541]",None,AIS,AIS,-35.616248,3.404552e-16,2024-01-03 23:55:40,69.0,"[42.26315, -83.10541]","[42.26315, -83.10539]",0.001646
918,164,316009457,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[42.26315, -83.10541]",None,AIS,AIS,-35.641895,3.318346e-16,2024-01-03 23:56:49,71.0,"[42.26315, -83.10541]","[42.26315, -83.10541]",0.0
919,164,316009457,2,2024-01-03 23:59:01,2024-01-03 23:59:01,NaT,"[42.26315, -83.1054]",None,AIS,AIS,-35.667542,3.234324e-16,2024-01-03 23:58:00,61.0,"[42.26315, -83.1054]","[42.26315, -83.10541]",0.000823
920,164,316009457,2,2024-01-04 00:00:00,2024-01-04 00:00:00,NaT,"[42.26315, -83.1054]",None,AIS,AIS,-35.693188,3.152429e-16,2024-01-03 23:59:01,59.0,"[42.26315, -83.1054]","[42.26315, -83.1054]",0.0


In [12]:
AIS_RF_probability(49.0, 1.356456)

np.float64(0.8889772304874483)

In [13]:
df_merged.to_csv("juiste_sequence.csv", index=False)

In [14]:
df_merged["delta_time_sec"].mean(), df_merged["distance_km"].mean()

(np.float64(93.73832790445168), np.float64(0.15264899839240148))

In [15]:
sequence = test_set[test_set['TrackID'] == 291]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

In [16]:
path, probability, log_probs = model.viterbi(AIS_sequence, RF_sequence)

In [17]:
df = test_set[test_set["TrackID"] == 164]
match_rows = df[df["State"] == "M"]

rf_rows = match_rows.copy()
rf_rows["AIS"] = None
rf_rows["AIS_Timestamp"] = pd.NaT
rf_rows["Timestamp"] = rf_rows["RF_Timestamp"]
rf_rows["State"] = "RF"

df2 = test_set[test_set["TrackID"] == 291]

match_rows = df2[df2["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

df2 = df2[(df2["State"] != "M") & (df2["State"] != "RF")]
df2 = pd.concat([df2, ais_rows, rf_rows[0:1]], ignore_index=True)
df2 = df2.sort_values(by=["Timestamp"]).reset_index(drop=True) 

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged = df2.copy().reset_index(drop=True)
df_merged["BestPathState"] = df_path["State"]
df_merged["LogProb"] = df_path["LogProb"]
df_merged["Probability"] = np.exp(df_path["LogProb"])

df_merged["prev_Timestamp"] = df_merged["Timestamp"].shift(1)
df_merged["delta_time_sec"] = (df_merged["Timestamp"] - df_merged["prev_Timestamp"]).dt.total_seconds()

df_merged["coord"] = df_merged["AIS"].combine_first(df_merged["RF"])
df_merged["prev_coord"] = df_merged["coord"].shift(1)
df_merged["distance_km"] = None  

for idx, row in df_merged.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged.at[idx, "distance_km"] = dist
df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km
0,291,366904890,2,2024-01-03 00:00:03,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None
1,291,366904890,2,2024-01-03 00:00:04,2024-01-03 00:00:04,NaT,"[42.27507, -83.10986]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:00:03,1.0,"[42.27507, -83.10986]",None,None
2,291,366904890,2,2024-01-03 00:01:06,2024-01-03 00:01:06,NaT,"[42.27466, -83.10859]",None,AIS,AIS,-1.124278,3.248868e-01,2024-01-03 00:00:04,62.0,"[42.27466, -83.10859]","[42.27507, -83.10986]",0.114003
3,291,366904890,2,2024-01-03 00:02:54,2024-01-03 00:02:54,NaT,"[42.27421, -83.10593]",None,AIS,AIS,-1.149946,3.166538e-01,2024-01-03 00:01:06,108.0,"[42.27421, -83.10593]","[42.27466, -83.10859]",0.224503
4,291,366904890,2,2024-01-03 00:03:57,2024-01-03 00:03:57,NaT,"[42.27449, -83.10459]",None,AIS,AIS,-1.175615,3.086291e-01,2024-01-03 00:02:54,63.0,"[42.27449, -83.10459]","[42.27421, -83.10593]",0.114563
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,291,366904890,2,2024-01-03 23:55:41,2024-01-03 23:55:41,NaT,"[45.07417, -83.10874]",None,AIS,AIS,-38.257853,2.425636e-17,2024-01-03 23:54:40,61.0,"[45.07417, -83.10874]","[45.07101, -83.10774]",0.360044
1021,291,366904890,2,2024-01-03 23:56:51,2024-01-03 23:56:51,NaT,"[45.0778, -83.10986]",None,AIS,AIS,-38.283500,2.364217e-17,2024-01-03 23:55:41,70.0,"[45.0778, -83.10986]","[45.07417, -83.10874]",0.413108
1022,291,366904890,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[45.08135, -83.11099]",None,AIS,AIS,-38.309146,2.304354e-17,2024-01-03 23:56:51,69.0,"[45.08135, -83.11099]","[45.0778, -83.10986]",0.404591
1023,291,366904890,2,2024-01-03 23:59:10,2024-01-03 23:59:10,NaT,"[45.08501, -83.11218]",None,AIS,AIS,-38.334793,2.246006e-17,2024-01-03 23:58:00,70.0,"[45.08501, -83.11218]","[45.08135, -83.11099]",0.417561


In [18]:
df = test_set[test_set["TrackID"] == 164]
match_rows = df[df["State"] == "M"]

ais_rows = match_rows.copy()
ais_rows["RF"] = None
ais_rows["RF_Timestamp"] = pd.NaT
ais_rows["State"] = "AIS"

rf_rows = match_rows.copy()
rf_rows["AIS"] = None
rf_rows["AIS_Timestamp"] = pd.NaT
rf_rows["Timestamp"] = rf_rows["RF_Timestamp"]
rf_rows["State"] = "RF"

df = df[(df["State"] != "M") & (df["State"] != "RF")]
df = pd.concat([df, ais_rows, rf_rows.iloc[0:1]], ignore_index=True)
df = df.sort_values(by=["Timestamp"]).reset_index(drop=True) 

df_path = pd.DataFrame({"State": path, "LogProb": log_probs})

df_merged = df.copy().reset_index(drop=True)
df_merged["BestPathState"] = df_path["State"]
df_merged["LogProb"] = df_path["LogProb"]
df_merged["Probability"] = np.exp(df_path["LogProb"])

df_merged["prev_Timestamp"] = df_merged["Timestamp"].shift(1)
df_merged["delta_time_sec"] = (df_merged["Timestamp"] - df_merged["prev_Timestamp"]).dt.total_seconds()

df_merged["coord"] = df_merged["AIS"].combine_first(df_merged["RF"])
df_merged["prev_coord"] = df_merged["coord"].shift(1)
df_merged["distance_km"] = None  

for idx, row in df_merged.iterrows():
    if row["coord"] is not None and row["prev_coord"] is not None:
        dist = haversine(row["prev_coord"], row["coord"], unit=Unit.KILOMETERS)
        df_merged.at[idx, "distance_km"] = dist
        
df_merged["prev_state"] = df_merged["State"].shift(1)
df_merged["transition"] = df_merged["prev_state"] + "→" + df_merged["State"]
df_merged["trans_prob"] = np.nan

for idx, row in df_merged.iterrows():
    trans = row["transition"]

    if row["prev_state"] == "begin":
        trans_prob = 0.3

    elif trans in ["AIS→AIS", "AIS→RF", "RF→RF", "RF→AIS"]:
        prev = df_merged.loc[idx - 1]

        if trans == "AIS→AIS":
            time = (row["AIS_Timestamp"] - prev["AIS_Timestamp"]).total_seconds()
            dist = haversine(row["AIS"], prev["AIS"], unit=Unit.KILOMETERS)
            if dist < 0.01 and time > 1800:
                trans_prob = 0.5
            else:
                trans_prob = AIS_RF_probability(time, dist)
        elif trans == "AIS→RF":
            time = row["RF_Timestamp"] - prev["AIS_Timestamp"]
            dist = haversine(row["RF"], prev["AIS"], unit=Unit.KILOMETERS)
            trans_prob = AIS_RF_probability(time.total_seconds(), dist)
        elif trans == "RF→RF":
            time = row["RF_Timestamp"] - prev["RF_Timestamp"]
            dist = haversine(row["RF"], prev["RF"], unit=Unit.KILOMETERS)
            trans_prob = AIS_RF_probability(time.total_seconds(), dist)
        elif trans == "RF→AIS":
            time = row["AIS_Timestamp"] - prev["RF_Timestamp"]
            dist = haversine(row["AIS"], prev["RF"], unit=Unit.KILOMETERS)
            trans_prob = AIS_RF_probability(time.total_seconds(), dist)

    elif row["State"] == "end":
        trans_prob = 0.05
    
    else:
        continue

    df_merged.at[idx, "trans_prob"] = trans_prob

df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
0,164,316009457,2,2024-01-03 00:01:08,NaT,NaT,None,None,begin,begin,0.000000,1.000000e+00,NaT,NaN,None,None,None,None,NaN,NaN
1,164,316009457,2,2024-01-03 00:01:09,2024-01-03 00:01:09,NaT,"[43.26082, -82.44656]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:01:08,1.0,"[43.26082, -82.44656]",None,None,begin,begin→AIS,0.300000
2,164,316009457,2,2024-01-03 00:02:59,2024-01-03 00:02:59,NaT,"[43.25506, -82.44536]",None,AIS,AIS,-1.124278,3.248868e-01,2024-01-03 00:01:09,110.0,"[43.25506, -82.44536]","[43.26082, -82.44656]",0.647814,AIS,AIS→AIS,0.886610
3,164,316009457,2,2024-01-03 00:04:09,2024-01-03 00:04:09,NaT,"[43.2514, -82.44465]",None,AIS,AIS,-1.149946,3.166538e-01,2024-01-03 00:02:59,70.0,"[43.2514, -82.44465]","[43.25506, -82.44536]",0.411016,AIS,AIS→AIS,0.926341
4,164,316009457,2,2024-01-03 00:05:20,2024-01-03 00:05:20,NaT,"[43.24775, -82.4439]",None,AIS,AIS,-1.175615,3.086291e-01,2024-01-03 00:04:09,71.0,"[43.24775, -82.4439]","[43.2514, -82.44465]",0.410383,AIS,AIS→AIS,0.925690
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
917,164,316009457,2,2024-01-03 23:56:49,2024-01-03 23:56:49,NaT,"[42.26315, -83.10541]",None,AIS,AIS,-35.616248,3.404552e-16,2024-01-03 23:55:40,69.0,"[42.26315, -83.10541]","[42.26315, -83.10539]",0.001646,AIS,AIS→AIS,0.950079
918,164,316009457,2,2024-01-03 23:58:00,2024-01-03 23:58:00,NaT,"[42.26315, -83.10541]",None,AIS,AIS,-35.641895,3.318346e-16,2024-01-03 23:56:49,71.0,"[42.26315, -83.10541]","[42.26315, -83.10541]",0.0,AIS,AIS→AIS,0.948766
919,164,316009457,2,2024-01-03 23:59:01,2024-01-03 23:59:01,NaT,"[42.26315, -83.1054]",None,AIS,AIS,-35.667542,3.234324e-16,2024-01-03 23:58:00,61.0,"[42.26315, -83.1054]","[42.26315, -83.10541]",0.000823,AIS,AIS→AIS,0.955773
920,164,316009457,2,2024-01-04 00:00:00,2024-01-04 00:00:00,NaT,"[42.26315, -83.1054]",None,AIS,AIS,-35.693188,3.152429e-16,2024-01-03 23:59:01,59.0,"[42.26315, -83.1054]","[42.26315, -83.1054]",0.0,AIS,AIS→AIS,0.957238


In [19]:
df_merged.sort_values(by=["trans_prob"], inplace=True)
df_merged

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,BestPathState,LogProb,Probability,prev_Timestamp,delta_time_sec,coord,prev_coord,distance_km,prev_state,transition,trans_prob
160,164,316009457,2,2024-01-03 05:29:40,2024-01-03 05:29:40,NaT,"[42.45249, -82.77025]",None,AIS,AIS,-16.201736,9.197617e-08,2024-01-03 03:04:39,8701.0,"[42.45249, -82.77025]","[42.75882, -82.46882]",42.057645,AIS,AIS→AIS,0.000127
921,164,316009457,2,2024-01-04 00:00:01,NaT,NaT,None,None,end,AIS,-35.718835,3.072608e-16,2024-01-04 00:00:00,1.0,None,"[42.26315, -83.1054]",None,AIS,AIS→end,0.050000
1,164,316009457,2,2024-01-03 00:01:09,2024-01-03 00:01:09,NaT,"[43.26082, -82.44656]",None,AIS,AIS,-1.098612,3.333333e-01,2024-01-03 00:01:08,1.0,"[43.26082, -82.44656]",None,None,begin,begin→AIS,0.300000
611,164,316009457,2,2024-01-03 18:03:20,2024-01-03 18:03:20,NaT,"[42.26352, -83.10525]",None,AIS,AIS,-27.768374,8.716624e-13,2024-01-03 14:12:39,13841.0,"[42.26352, -83.10525]","[42.26351, -83.10523]",0.001986,AIS,AIS→AIS,0.500000
187,164,316009457,2,2024-01-03 06:02:20,2024-01-03 06:02:20,NaT,"[42.38829, -82.8656]",None,AIS,AIS,-16.894196,4.601972e-08,2024-01-03 05:59:41,159.0,"[42.38829, -82.8656]","[42.39352, -82.85779]",0.865786,AIS,AIS→AIS,0.843897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340,164,316009457,2,2024-01-03 09:00:00,2024-01-03 09:00:00,NaT,"[42.26334, -83.10528]",None,AIS,AIS,-20.818133,9.094939e-10,2024-01-03 08:59:30,30.0,"[42.26334, -83.10528]","[42.26334, -83.1053]",0.001646,AIS,AIS→AIS,0.977926
548,164,316009457,2,2024-01-03 13:00:00,2024-01-03 13:00:00,NaT,"[42.2635, -83.10525]",None,AIS,AIS,-26.152635,4.385860e-12,2024-01-03 12:59:31,29.0,"[42.2635, -83.10525]","[42.26351, -83.10525]",0.001112,AIS,AIS→AIS,0.978682
104,164,316009457,2,2024-01-03 02:00:00,2024-01-03 02:00:00,NaT,"[42.93295, -82.45304]",None,AIS,AIS,-14.765524,3.867358e-07,2024-01-03 01:59:40,20.0,"[42.93295, -82.45304]","[42.93383, -82.45251]",0.106942,AIS,AIS→AIS,0.978992
393,164,316009457,2,2024-01-03 10:00:00,2024-01-03 10:00:00,NaT,"[42.26326, -83.10537]",None,AIS,AIS,-22.177405,2.336013e-10,2024-01-03 09:59:51,9.0,"[42.26326, -83.10537]","[42.26326, -83.10537]",0.0,AIS,AIS→AIS,0.993356


In [22]:
test_set[(test_set["TrackID"] == 164) & (test_set["Timestamp"] == "2024-01-03 18:03:20")].index

Index([103715], dtype='int64')

In [23]:
test_set.iloc[103715-3:103715+3]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
103712,164,316009457,2,2024-01-03 14:10:20,2024-01-03 14:10:20,NaT,"[42.26348, -83.10523]",None,AIS
103713,164,316009457,2,2024-01-03 14:11:30,2024-01-03 14:11:30,NaT,"[42.26349, -83.10524]",None,AIS
103714,164,316009457,2,2024-01-03 14:12:39,2024-01-03 14:12:39,NaT,"[42.26351, -83.10523]",None,AIS
103715,164,316009457,2,2024-01-03 18:03:20,2024-01-03 18:03:20,NaT,"[42.26352, -83.10525]",None,AIS
103716,164,316009457,2,2024-01-03 18:04:31,2024-01-03 18:04:31,NaT,"[42.2635, -83.10525]",None,AIS
103717,164,316009457,2,2024-01-03 18:05:39,2024-01-03 18:05:39,NaT,"[42.26352, -83.10525]",None,AIS


In [24]:
AIS_RF_probability(110.0, 0.647814)

np.float64(0.8866097838531733)

In [25]:
0.1

0.1

In [26]:
-(35/1025)

-0.03414634146341464

In [27]:
sequence3 = test_set[test_set['TrackID'] == 100]

AIS_sequence3 = [(time, coord) for time, coord in zip(sequence3["AIS_Timestamp"], sequence3["AIS"]) 
                if pd.notna(time) and coord is not None]

RF_sequence3 = [(time, coord) for time, coord in zip(sequence3["RF_Timestamp"], sequence3["RF"]) 
               if pd.notna(time) and coord is not None]

In [28]:
path3, probability3, log_probs3 = model.viterbi(AIS_sequence3, RF_sequence3)

In [29]:
RF_M_indexes_pred_path3 = [(i, state) for i, state in enumerate(path3) if state in {"RF", "M"}]
RF_M_indexes_real_path3 = [(i, state) for i, state in enumerate(sequence3["State"]) if state in {"RF", "M"}]
print(f"The indexes of RF and/or M of predicted path are: {RF_M_indexes_pred_path3}, \
      \nThe indexes of RF and/or M of the real path are: {RF_M_indexes_real_path3}")

The indexes of RF and/or M of predicted path are: [(110, 'RF'), (286, 'RF')],       
The indexes of RF and/or M of the real path are: [(110, 'RF'), (286, 'RF')]


In [30]:
sequence = test_set[test_set['TrackID'] == 291]

AIS_sequence = [(time, coord) for time, coord in zip(sequence["AIS_Timestamp"], sequence["AIS"]) 
                if pd.notna(time) and coord is not None]

AIS_sequence

[(Timestamp('2024-01-03 00:00:04'), array([ 42.27507, -83.10986])),
 (Timestamp('2024-01-03 00:01:06'), array([ 42.27466, -83.10859])),
 (Timestamp('2024-01-03 00:02:54'), array([ 42.27421, -83.10593])),
 (Timestamp('2024-01-03 00:03:57'), array([ 42.27449, -83.10459])),
 (Timestamp('2024-01-03 00:05:00'), array([ 42.2751 , -83.10387])),
 (Timestamp('2024-01-03 00:06:03'), array([ 42.27593, -83.10332])),
 (Timestamp('2024-01-03 00:07:14'), array([ 42.27718, -83.10256])),
 (Timestamp('2024-01-03 00:08:22'), array([ 42.27871, -83.10181])),
 (Timestamp('2024-01-03 00:09:33'), array([ 42.28057, -83.10091])),
 (Timestamp('2024-01-03 00:10:43'), array([ 42.2826 , -83.09994])),
 (Timestamp('2024-01-03 00:11:53'), array([ 42.28475, -83.09887])),
 (Timestamp('2024-01-03 00:13:03'), array([ 42.28697, -83.09778])),
 (Timestamp('2024-01-03 00:14:13'), array([ 42.28917, -83.09656])),
 (Timestamp('2024-01-03 00:15:23'), array([ 42.29139, -83.09532])),
 (Timestamp('2024-01-03 00:16:33'), array([ 42.2